# Monte Carlo Pricing for Barrier and Early-Exercise Options


**Models:** Heston (Stochastic Volatility) & Merton (Jump-Diffusion)  
**Method:** Monte-Carlo (200,000 paths, 90 time steps)

In [9]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# General Parameters 
S0    = 80.0
r     = 0.055
sigma = 0.35
T     = 3/12
K_atm = 80.0

# Heston parameters
v0    = 0.032
kappa = 1.85
theta = 0.045
xi    = sigma     # vol-of-vol

# Merton parameters
mu_J    = -0.5
delta_J = 0.22

N_PATHS = 200_000
N_STEPS = 90
dt      = T / N_STEPS
print('Parameters loaded.')

Parameters loaded.


In [10]:
# Simulation Engines

def heston_paths(S0, r, v0, kappa, theta, xi, rho, T, N_PATHS, N_STEPS):
    dt = T / N_STEPS
    S = np.full(N_PATHS, S0)
    v = np.full(N_PATHS, v0)
    path_S = np.zeros((N_STEPS + 1, N_PATHS))
    path_S[0] = S0
    for t in range(N_STEPS):
        Z1 = np.random.standard_normal(N_PATHS)
        Z2 = rho * Z1 + np.sqrt(1 - rho**2) * np.random.standard_normal(N_PATHS)
        v_pos = np.maximum(v, 0)
        S = S * np.exp((r - 0.5 * v_pos) * dt + np.sqrt(v_pos * dt) * Z1)
        v = v + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos * dt) * Z2
        v = np.maximum(v, 0)
        path_S[t+1] = S
    return path_S


def merton_paths(S0, r, sigma, lam, mu_J, delta_J, T, N_PATHS, N_STEPS):
    dt = T / N_STEPS
    k  = np.exp(mu_J + 0.5 * delta_J**2) - 1   # compensator
    S  = np.full(N_PATHS, S0, dtype=float)
    path_S = np.zeros((N_STEPS + 1, N_PATHS))
    path_S[0] = S0
    for t in range(N_STEPS):
        Z      = np.random.standard_normal(N_PATHS)
        N_jump = np.random.poisson(lam * dt, N_PATHS)
        J      = np.zeros(N_PATHS)
        for i in range(N_PATHS):
            if N_jump[i] > 0:
                J[i] = np.sum(np.random.normal(mu_J, delta_J, N_jump[i]))
        S = S * np.exp((r - lam*k - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z + J)
        path_S[t+1] = S
    return path_S


def price_european(payoff_fn, paths, r, T):
    return np.exp(-r * T) * np.mean(payoff_fn(paths[-1]))


def price_american_call_lsm(paths, K, r, T, N_STEPS):
    """Longstaff-Schwartz LSM for American call."""
    dt = T / N_STEPS
    cash_flow = np.maximum(paths[-1] - K, 0).astype(float)
    for t in range(N_STEPS - 1, 0, -1):
        S_t       = paths[t]
        intrinsic = np.maximum(S_t - K, 0)
        itm       = intrinsic > 0
        if np.sum(itm) > 10:
            X     = S_t[itm]
            Y     = cash_flow[itm] * np.exp(-r * dt)
            basis = np.column_stack([np.ones_like(X), X, X**2])
            try:
                coeffs, _, _, _ = np.linalg.lstsq(basis, Y, rcond=None)
                cont  = basis @ coeffs
                ex    = intrinsic[itm] > cont
                cash_flow[itm]  = np.where(ex, intrinsic[itm], cash_flow[itm] * np.exp(-r * dt))
                cash_flow[~itm] *= np.exp(-r * dt)
            except:
                cash_flow *= np.exp(-r * dt)
        else:
            cash_flow *= np.exp(-r * dt)
    return np.mean(cash_flow)


def price_barrier_call_ui(paths, K, B, r, T):
    """Up-and-in call: payoff only if max(path) >= B."""
    activated = np.max(paths, axis=0) >= B
    payoff    = np.where(activated, np.maximum(paths[-1] - K, 0), 0)
    return np.exp(-r * T) * np.mean(payoff)


def price_barrier_put_di(paths, K, B, r, T):
    """Down-and-in put: payoff only if min(path) <= B."""
    activated = np.min(paths, axis=0) <= B
    payoff    = np.where(activated, np.maximum(K - paths[-1], 0), 0)
    return np.exp(-r * T) * np.mean(payoff)


print('Simulation engines defined.')

Simulation engines defined.


In [11]:
# Generate Base Paths
print('Generating paths (this takes ~1-2 minutes)...')

np.random.seed(42)
paths_heston_30 = heston_paths(S0, r, v0, kappa, theta, xi, -0.30, T, N_PATHS, N_STEPS)
print('  Heston rho=-0.30 done')

np.random.seed(42)
paths_heston_70 = heston_paths(S0, r, v0, kappa, theta, xi, -0.70, T, N_PATHS, N_STEPS)
print('  Heston rho=-0.70 done')

np.random.seed(42)
paths_merton_75 = merton_paths(S0, r, sigma, 0.75, mu_J, delta_J, T, N_PATHS, N_STEPS)
print('  Merton lambda=0.75 done')

np.random.seed(42)
paths_merton_25 = merton_paths(S0, r, sigma, 0.25, mu_J, delta_J, T, N_PATHS, N_STEPS)
print('  Merton lambda=0.25 done')

print('All paths ready.')

Generating paths (this takes ~1-2 minutes)...
  Heston rho=-0.30 done
  Heston rho=-0.70 done
  Merton lambda=0.75 done
  Merton lambda=0.25 done
All paths ready.


## Put-Call Parity Check
Put-call parity states: **C − P = S₀ − K·e^(−rT)**

We verify this for all four model/parameter combinations across both stochastic-volatility and jump-diffusion parameter sets.

In [12]:
call_fn = lambda S: np.maximum(S - K_atm, 0)
put_fn  = lambda S: np.maximum(K_atm - S, 0)

# Prices from , , , 
c_h30 = price_european(call_fn, paths_heston_30, r, T)
p_h30 = price_european(put_fn,  paths_heston_30, r, T)
c_h70 = price_european(call_fn, paths_heston_70, r, T)
p_h70 = price_european(put_fn,  paths_heston_70, r, T)
c_m75 = price_european(call_fn, paths_merton_75, r, T)
p_m75 = price_european(put_fn,  paths_merton_75, r, T)
c_m25 = price_european(call_fn, paths_merton_25, r, T)
p_m25 = price_european(put_fn,  paths_merton_25, r, T)

pcp_rhs = S0 - K_atm * np.exp(-r * T)   # 1.0925

rows = []
for label, c, p in [
    ('Heston  ρ=−0.30', c_h30, p_h30),
    ('Heston  ρ=−0.70', c_h70, p_h70),
    ('Merton  λ=0.75', c_m75, p_m75),
    ('Merton  λ=0.25', c_m25, p_m25),
]:
    lhs  = c - p
    diff = abs(lhs - pcp_rhs)
    ok   = 'YES' if diff < 0.10 else 'APPROX' if diff < 0.30 else 'NO'
    rows.append({'Model / Params': label,
                 'Call (C)': round(c, 2),
                 'Put (P)': round(p, 2),
                 'C − P': round(lhs, 4),
                 'S₀ − K·e^{−rT}': round(pcp_rhs, 4),
                 '|Diff|': round(diff, 4),
                 'Parity Holds?': ok})

df11 = pd.DataFrame(rows).set_index('Model / Params')
df11

,Call (C),Put (P),C − P,S₀ − K·e^{−rT},|Diff|,Parity Holds?
Model / Params,,,,,,
Heston ρ=−0.30 [Q5],3.47,2.38,1.0824,1.0925,0.0100,YES
Heston ρ=−0.70 [Q6],3.49,2.41,1.0825,1.0925,0.0099,YES
Merton λ=0.75 [Q8],8.31,7.19,1.1247,1.0925,0.0322,YES
Merton λ=0.25 [Q9],6.82,5.72,1.0971,1.0925,0.0047,YES


### Findings

Put-call parity requires **C − P = S₀ − K·e^{−rT}**. For our parameters, the theoretical right-hand side equals **\$1.0925**.

| Model | C − P | Theoretical | |Diff| |
|---|---|---|---|
| Heston ρ=−0.30 | 1.0824 | 1.0925 | 0.0100 |
| Heston ρ=−0.70 | 1.0825 | 1.0925 | 0.0099 |
| Merton λ=0.75 | 1.1247 | 1.0925 | 0.0322 |
| Merton λ=0.25 | 1.0971 | 1.0925 | 0.0047 |

**All four models satisfy put-call parity** (deviations < \$0.04). The small residual differences are entirely attributable to **Monte-Carlo sampling error** - they diminish as the number of simulation paths increases. This result is expected: put-call parity is a model-free no-arbitrage relationship that must hold under any risk-neutral pricing framework, including Heston and Merton, as long as European options are priced using the same underlying simulated paths and the same risk-neutral measure.

## Volatility Smile Across 7 Strikes

Strike = S₀ / moneyness. Moneyness values: 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15.

In [13]:
moneyness_vals = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
strikes        = [round(S0 / m, 2) for m in moneyness_vals]

rows12 = []
for K_i, m_i in zip(strikes, moneyness_vals):
    cf = lambda S, K=K_i: np.maximum(S - K, 0)
    rows12.append({
        'Moneyness': m_i,
        'Strike K': K_i,
        'Type': 'OTM Call' if m_i < 1 else ('ATM' if m_i == 1 else 'ITM Call'),
        'Heston ρ=−0.30': round(price_european(cf, paths_heston_30, r, T), 2),
        'Heston ρ=−0.70': round(price_european(cf, paths_heston_70, r, T), 2),
        'Merton λ=0.75':  round(price_european(cf, paths_merton_75, r, T), 2),
        'Merton λ=0.25':  round(price_european(cf, paths_merton_25, r, T), 2),
    })

df12 = pd.DataFrame(rows12).set_index('Strike K')
df12

,Moneyness,Type,Heston ρ=−0.30,Heston ρ=−0.70,Merton λ=0.75,Merton λ=0.25
Strike K,,,,,,
94.12,0.85,OTM Call,0.14,0.04,2.82,1.98
88.89,0.90,OTM Call,0.55,0.38,4.36,3.26
84.21,0.95,OTM Call,1.60,1.49,6.22,4.89
80.00,1.00,ATM,3.47,3.49,8.31,6.82
76.19,1.05,ITM Call,5.97,6.06,10.55,8.96
72.73,1.10,ITM Call,8.77,8.86,12.84,11.23
69.57,1.15,ITM Call,11.60,11.67,15.12,13.55


### Findings

The table above reports European call prices across 7 strikes (3 OTM, 1 ATM, 3 ITM).

## Key observations:

1. **Call prices decrease as the strike rises** (for a fixed spot), which is consistent with basic option theory - OTM calls (high K, low moneyness) are worth less than ITM calls (low K, high moneyness).

2. **Heston model (ρ = −0.30 vs ρ = −0.70):** The two correlation values produce very similar call prices across all strikes. The more negative correlation (−0.70) generates a slightly stronger volatility skew - OTM calls (high K) are priced a touch lower and ITM calls a touch higher - reflecting the asymmetric impact of negative equity-vol correlation, consistent with a volatility *skew* (not a symmetric smile).

3. **Merton model (λ = 0.75 vs λ = 0.25):** Merton prices are substantially higher than Heston prices across all strikes. This is driven by the jump component: with μ_J = −0.5, jumps are predominantly **downward**, which adds extra variance to the distribution and inflates option premiums. Higher jump intensity (λ = 0.75) amplifies this effect relative to λ = 0.25.

4. **The volatility smile / skew pattern** is present in both models, with deeper OTM options showing the characteristic pricing behaviour associated with fat tails and skewness beyond Black-Scholes.

## American Call Option (Heston, ρ = −0.30)

We price the ATM American call using the **Longstaff-Schwartz (LSM)** algorithm and compare the price and greeks to the European call from the European baseline.

In [14]:
def heston_american_call(S0_val, rho=-0.30, seed=42):
    np.random.seed(seed)
    p = heston_paths(S0_val, r, v0, kappa, theta, xi, rho, T, N_PATHS, N_STEPS)
    return price_american_call_lsm(p, K_atm, r, T, N_STEPS)

def heston_euro_call(S0_val, rho=-0.30, seed=42):
    np.random.seed(seed)
    p = heston_paths(S0_val, r, v0, kappa, theta, xi, rho, T, N_PATHS, N_STEPS)
    return price_european(lambda S: np.maximum(S - K_atm, 0), p, r, T)

eps = 0.5

# American call price and greeks
am_mid   = heston_american_call(S0)
am_up    = heston_american_call(S0 + eps)
am_down  = heston_american_call(S0 - eps)
am_delta = (am_up - am_down) / (2 * eps)
am_gamma = (am_up - 2*am_mid + am_down) / eps**2

# European call price and greeks ()
eu_mid   = heston_euro_call(S0)
eu_up    = heston_euro_call(S0 + eps)
eu_down  = heston_euro_call(S0 - eps)
eu_delta = (eu_up - eu_down) / (2 * eps)
eu_gamma = (eu_up - 2*eu_mid + eu_down) / eps**2

df13 = pd.DataFrame({
    'Option Type':        ['European Call ()', 'American Call ()'],
    'Price':              [round(eu_mid, 2),   round(am_mid, 2)],
    'Delta':              [round(eu_delta, 4), round(am_delta, 4)],
    'Gamma':              [round(eu_gamma, 4), round(am_gamma, 4)],
    'Early Exercise Prem':['-',  round(am_mid - eu_mid, 2)],
}).set_index('Option Type')
df13

,Price,Delta,Gamma,Early Exercise Prem
Option Type,,,,
European Call (Q5),3.47,0.6033,0.0548,-
American Call (Q13),3.43,0.6000,0.0595,-0.04


### Findings

## Prices:
The American call price (≈ \$3.43) is essentially equal to - or in some simulations marginally below - the European call price (≈ \$3.47). The **early exercise premium is approximately \$0.00 to −\$0.04**, which is consistent with a well-known theoretical result: **for a non-dividend-paying stock, it is never optimal to exercise an American call early**. This is because the holder always does better to sell the option rather than exercise, since the option embeds time value that would be destroyed upon early exercise. The small numerical difference (including the slightly negative value from LSM) is within Monte-Carlo simulation noise.

## Greeks - Delta:
Both American and European calls produce a delta of approximately **0.60**, consistent with an at-the-money option under the Heston model. The values are very close because - again - the American call is essentially never exercised early on a non-dividend-paying stock, making its hedging behaviour indistinguishable from the European call.

## Greeks - Gamma:
Gamma is also very similar for both option types (~0.055–0.060), measuring the curvature of the price as a function of the underlying. Any small differences are due to Monte-Carlo noise in the LSM backward-induction procedure.

**Conclusion:** Under the Heston model with no dividends, the American call and European call are effectively **equivalent** in price and Greeks. The LSM algorithm correctly identifies that early exercise is suboptimal at every time step.

## European Up-and-In Call (Heston, ρ = −0.70)

**Specification:** Barrier B = \$95, Strike K = \$95. The option becomes alive only if the stock price **reaches \$95 at any point** before maturity (even if it ends below \$95 at expiry). We use  Heston paths (ρ = −0.70).

In [15]:
B14  = 95.0
K14  = 95.0

# Up-and-in call
cui_price = price_barrier_call_ui(paths_heston_70, K14, B14, r, T)

# Standard European call (same K=95)
euro_call_K95 = price_european(lambda S: np.maximum(S - K14, 0), paths_heston_70, r, T)

# Barrier activation rate
activation_rate = np.mean(np.max(paths_heston_70, axis=0) >= B14)

df14 = pd.DataFrame({
    'Option': ['European Call (K=95)', 'Up-and-In Call (B=95, K=95)'],
    'Price':  [round(euro_call_K95, 2), round(cui_price, 2)],
    'Notes':  ['Standard vanilla call', f'Activated in {activation_rate*100:.2f}% of paths']
}).set_index('Option')
df14

,Price,Notes
Option,,
European Call (K=95),0.03,Standard vanilla call
"Up-and-In Call (B=95, K=95)",0.03,Activated in 2.19% of paths


### Findings

**The up-and-in call (CUI) is priced at ≈ \$0.03**, which equals the plain European call at the same strike of \$95.

## Why are they essentially equal?
With S₀ = \$80 and K = B = \$95, the barrier is **18.75% above the current stock price** over a 3-month horizon. Under the Heston model with the given parameters, only about **2.2% of simulated paths** ever reach the \$95 barrier. 

The critical insight is: **the CUI price equals the European call price** when K = B, because:
- If the path **never** reaches the barrier → the option is dead and pays nothing → but also the European call at K = \$95 would also pay nothing (since max price < \$95 < K).
- If the path **does** reach the barrier → the CUI is activated and pays max(S_T − 95, 0), which is **exactly** what the European call pays.

Therefore, when **K = B**, the up-and-in call and the standard European call are **theoretically identical** - both require the stock to end above \$95 to have any payoff, and the CUI's knock-in condition is always already implied by the European call being in-the-money. This is a known analytical result: *CUI(K, B=K) = European Call(K)*.

The small price of ≈ \$0.03 simply reflects the very low probability of the stock reaching \$95 in 3 months from \$80.

## European Down-and-In Put (Merton, λ = 0.75)

**Specification:** Barrier B = \$65, Strike K = \$65. The option becomes alive only if the stock price **reaches \$65 at any point** before maturity. We use Merton paths (λ = 0.75).

In [16]:
B15 = 65.0
K15 = 65.0

# Down-and-in put
pdi_price = price_barrier_put_di(paths_merton_75, K15, B15, r, T)

# Standard European put (same K=65)
euro_put_K65 = price_european(lambda S: np.maximum(K15 - S, 0), paths_merton_75, r, T)

# Barrier activation rate
activation_rate_put = np.mean(np.min(paths_merton_75, axis=0) <= B15)

df15 = pd.DataFrame({
    'Option': ['European Put (K=65)', 'Down-and-In Put (B=65, K=65)'],
    'Price':  [round(euro_put_K65, 2), round(pdi_price, 2)],
    'Notes':  ['Standard vanilla put', f'Activated in {activation_rate_put*100:.2f}% of paths']
}).set_index('Option')
df15

,Price,Notes
Option,,
European Put (K=65),2.75,Standard vanilla put
"Down-and-In Put (B=65, K=65)",2.75,Activated in 25.62% of paths


### Findings

**The down-and-in put (PDI) is priced at ≈ \$2.75**, which equals the plain European put at K = \$65.

## Why are they essentially equal?
With S₀ = \$80 and K = B = \$65, the barrier is **18.75% below the current stock price**. Under the Merton model with λ = 0.75 (high jump intensity with μ_J = −0.5 meaning predominantly downward jumps), approximately **25.6% of paths** touch the \$65 barrier - significantly more than the up-and-in case in , because the model features large negative jumps.

Nevertheless, the **PDI price equals the European put price** for the same structural reason as : when **K = B**, the knock-in condition and the vanilla put's in-the-money condition are equivalent:
- If min(S) **never** drops to \$65 → PDI is never activated → but also the European put with K = \$65 can only pay off if S_T < \$65, which would imply the path went below \$65 at some point, i.e., the barrier was already hit.
- If min(S) **does** reach \$65 → PDI is activated → and pays max(65 − S_T, 0), exactly like the European put.

Therefore, *PDI(K, B=K) = European Put(K)* - another known theoretical equivalence: the down-and-in put with B = K is equivalent to the standard European put. The result is confirmed numerically with a difference of ≈ \$0.00.

The price of ≈ \$2.75 is relatively modest because \$65 is 18.75% OTM for the put, even though the Merton model's large negative jumps provide more probability of reaching this level than a pure diffusion model would.